## tl;dr

Move durable lead storage to the first valid home-value contact step. A read-only Production query found one tightly time-adjacent, unclassified sequence with one funnel start, one address submission, one contact submission, and zero durable lead events. This is a credible missed-opportunity signal, not a verified public prospect. Source inspection independently confirms the structural gap: the released UI records `contact_submitted` before a separate required-phone screen, while `/api/leads` is not called until that later screen.

## Context & Methods

Reader: product and growth operators deciding whether to change the canonical home-value funnel. The source is the canonical Neon `public.analytics_events` ledger on the Production branch, queried read-only on 2026-08-24. The companion SQL is bounded to a 15-minute observation window and returns aggregate event counts only.

### Key Assumptions

- Registered QA/test UTM markers are excluded, but historical anonymous events do not carry `is_test`.
- The four stages are inferred to be one flow from event order and sub-16-second adjacency; session IDs are null, so this is not identity-level proof.
- Canonical event names are counted once; duplicate aliases such as `home_value_started` and `address_submit` are intentionally excluded.
- No conversion rate is calculated because page views and unique sessions are not reliable for this historical sequence.

## Data

The snapshot below reproduces the reviewed aggregate rows returned by `home_value_completion_integrity.sql`. It contains no contact details, address, IP, raw user agent, click ID, provider identifier, token, or credential.

In [1]:
from pathlib import Path
from pprint import pprint

sql_candidates = [
    Path('home_value_completion_integrity.sql'),
    Path('docs/phase9/analysis/home_value_completion_integrity.sql'),
]
sql_path = next(path for path in sql_candidates if path.exists())
source_sql = sql_path.read_text(encoding='utf-8')
observation_start = '2026-08-22T19:04:50Z'
observation_end = '2026-08-22T19:20:00Z'
snapshot_reviewed_on = '2026-08-24'

stages = [
    {'stage_order': 1, 'stage': 'Funnel start', 'event_name': 'funnel_started', 'events': 1},
    {'stage_order': 2, 'stage': 'Address submitted', 'event_name': 'address_submitted', 'events': 1},
    {'stage_order': 3, 'stage': 'Contact submitted', 'event_name': 'contact_submitted', 'events': 1},
    {'stage_order': 4, 'stage': 'Durable lead created', 'event_name': 'lead_created', 'events': 0},
]

print(f'SQL source: {sql_path}')
print(f'Observation window: {observation_start} to {observation_end}')
print(f'Snapshot reviewed: {snapshot_reviewed_on}')
pprint(stages)

SQL source: home_value_completion_integrity.sql
Observation window: 2026-08-22T19:04:50Z to 2026-08-22T19:20:00Z
Snapshot reviewed: 2026-08-24
[{'event_name': 'funnel_started',
  'events': 1,
  'stage': 'Funnel start',
  'stage_order': 1},
 {'event_name': 'address_submitted',
  'events': 1,
  'stage': 'Address submitted',
  'stage_order': 2},
 {'event_name': 'contact_submitted',
  'events': 1,
  'stage': 'Contact submitted',
  'stage_order': 3},
 {'event_name': 'lead_created',
  'events': 0,
  'stage': 'Durable lead created',
  'stage_order': 4}]


In [2]:
assert source_sql.lstrip().startswith('-- Read-only Production evidence query')
assert [row['stage_order'] for row in stages] == [1, 2, 3, 4]
assert all(isinstance(row['events'], int) and row['events'] >= 0 for row in stages)
assert stages[2]['events'] >= stages[3]['events']
print('Input validation: PASS')

Input validation: PASS


## Results

In [3]:
header = f"{'Stage':<24} {'Event':<22} {'Count':>5}"
print(header)
print('-' * len(header))
for row in stages:
    print(f"{row['stage']:<24} {row['event_name']:<22} {row['events']:>5}")

Stage                    Event                  Count
-----------------------------------------------------
Funnel start             funnel_started             1
Address submitted        address_submitted          1
Contact submitted        contact_submitted          1
Durable lead created     lead_created               0


In [4]:
contact_submissions = next(row['events'] for row in stages if row['event_name'] == 'contact_submitted')
durable_leads = next(row['events'] for row in stages if row['event_name'] == 'lead_created')
observed_storage_gap = contact_submissions - durable_leads

print(f'Observed contact submissions: {contact_submissions}')
print(f'Observed durable lead events: {durable_leads}')
print(f'Observed storage gap: {observed_storage_gap}')
assert observed_storage_gap == 1

Observed contact submissions: 1
Observed durable lead events: 0
Observed storage gap: 1


In [5]:
print('Stage progression (event counts, not unique people)')
for row in stages:
    bar = '█' * row['events'] if row['events'] else '·'
    print(f"{row['stage']:<24} {bar} {row['events']}")

Stage progression (event counts, not unique people)
Funnel start             █ 1
Address submitted        █ 1
Contact submitted        █ 1
Durable lead created     · 0


## Takeaways

1. The evidence supports eliminating the separate required-phone gate and storing the lead when name plus valid email are first submitted.
2. Phone can remain optional and, when supplied, continues through the same validation, consent, scoring, routing, deduplication, and notification path.
3. Add a privacy-safe `lead_submit_failed` event so future durable-write failures are visible without copying provider errors or lead data into analytics.
4. Do not call this sequence a live prospect. The correct label is an unclassified Production contact-step completion.